# 1. Comparison models: XGB and ENet 80/20 results

This notebook compares saved `.h5` outputs across:

- `results_xgb/`
- `res_xgb_zbig/`
- `results_enet/`
- `res_enet_zbig/`

It scans all result files, prints concise summaries, creates clean plots, applies configurable thresholds for "well predicted" proteins, and builds three horizontal partition plots showing which run is best per protein according to Pearson, cosine similarity, and MSE.

In [ ]:
# >>> pRNA figure routing (added by the pRNA update; keep as the first code cell)
# Every figure is shown inline. Every file written with savefig (png, pdf, svg) is redirected to
# figures_pdf/<notebook name>/ next to this notebook; sub-folders of the original path are preserved.
import os as _fig_os
from pathlib import Path as _FigPath
import matplotlib as _fig_mpl
import matplotlib.pyplot as _fig_plt
from matplotlib.figure import Figure as _FigFigure

NOTEBOOK_NAME = "1. comparison_models"
FIG_DIR = (_FigPath.cwd() / "figures_pdf" / NOTEBOOK_NAME).resolve()
FIG_DIR.mkdir(parents=True, exist_ok=True)


def _route_figure_path(fname):
    p = _FigPath(_fig_os.fspath(fname))
    full = p if p.is_absolute() else (_FigPath.cwd() / p)
    try:
        full.resolve().relative_to(FIG_DIR)
        return fname                                  # already inside this notebook's figure folder
    except ValueError:
        pass
    try:
        rel = _FigPath(_fig_os.path.relpath(full.resolve(), _FigPath.cwd()))
    except ValueError:                                # e.g. another drive on Windows
        rel = _FigPath(p.name)
    parts = [x for x in rel.parts if x not in ("..", ".", "figures_pdf", NOTEBOOK_NAME)]
    target = FIG_DIR.joinpath(*parts) if parts else FIG_DIR / p.name
    target.parent.mkdir(parents=True, exist_ok=True)
    return target


if not getattr(_FigFigure.savefig, "_prna_routed", False):
    _fig_orig_savefig = _FigFigure.savefig

    def _fig_routed_savefig(self, fname, *args, **kwargs):
        if isinstance(fname, (str, bytes, _FigPath)) or hasattr(fname, "__fspath__"):
            fname = _route_figure_path(fname)
        return _fig_orig_savefig(self, fname, *args, **kwargs)

    _fig_routed_savefig._prna_routed = True
    _FigFigure.savefig = _fig_routed_savefig

    _fig_orig_close = _fig_plt.close

    def _fig_close_and_show(fig=None):
        """Display figures that are closed before Jupyter could render them, then close."""
        import inspect as _fig_inspect
        _caller = _fig_inspect.currentframe().f_back
        _caller_mod = (_caller.f_globals.get("__name__", "") if _caller is not None else "")
        if _caller_mod.startswith(("matplotlib", "ipykernel")):   # backend housekeeping, not user code
            return _fig_orig_close(fig)
        if "inline" in str(_fig_mpl.get_backend()).lower():
            from IPython.display import display as _fig_display
            open_nums = set(_fig_plt.get_fignums())
            if fig is None:
                figs = [_fig_plt.gcf()] if open_nums else []
            elif isinstance(fig, str) and fig == "all":
                figs = [_fig_plt.figure(n) for n in sorted(open_nums)]
            elif isinstance(fig, _FigFigure):
                figs = [fig] if getattr(fig, "number", None) in open_nums else []
            elif isinstance(fig, int):
                figs = [_fig_plt.figure(fig)] if fig in open_nums else []
            else:
                figs = []
            for _f in figs:
                if not getattr(_f, "_prna_displayed", False):
                    _f._prna_displayed = True
                    _fig_display(_f)
        return _fig_orig_close(fig)

    _fig_plt.close = _fig_close_and_show

    _fig_orig_show = _fig_plt.show

    def _fig_show_and_mark(*args, **kwargs):
        for _n in _fig_plt.get_fignums():                    # shown now -> never re-displayed on close
            _fig_plt.figure(_n)._prna_displayed = True
        return _fig_orig_show(*args, **kwargs)

    _fig_plt.show = _fig_show_and_mark

print("Figures are saved to", FIG_DIR)

In [ ]:
from pathlib import Path
import json
import lzma
import pickle
import re
import warnings

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

ROOT = Path.cwd()
RESULT_FOLDERS = {
    "xgb_base": ROOT / "results_xgb",
    "xgb_zbig": ROOT / "res_xgb_zbig",
    "enet_base": ROOT / "results_enet",
    "enet_zbig": ROOT / "res_enet_zbig",
}

OUT_DIR = ROOT / "comparison_outputs"
OUT_DIR.mkdir(exist_ok=True)

# Thresholds for defining "well predicted" proteins. Adjust as needed.
PEARSON_GOOD_THRESHOLD = 0.70
COSINE_GOOD_THRESHOLD = 0.70
# If None, the notebook uses the 25th percentile of per-protein best MSE as a data-driven cutoff.
MSE_GOOD_THRESHOLD = None

print("Project root:", ROOT)
print("Output folder:", OUT_DIR)
for name, folder in RESULT_FOLDERS.items():
    print(f"{name:10s}: {folder} | exists={folder.exists()}")

In [ ]:
# Shared CCLE validation utilities
from pathlib import Path as _Path
import sys as _sys

# Make src/utils_ccle.py importable from repo root, notebooks/, notebooks2/, or this folder.
for _candidate in [
    _Path.cwd(),
    _Path.cwd().parent,
]:
    if (_candidate / "src" / "utils_ccle.py").exists():
        if str(_candidate) not in _sys.path:
            _sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError("Could not find src/utils_ccle.py from the current notebook location.")

import src.utils_ccle as uc
from src.utils_ccle import *
uc.set_module_globals(globals())
print("Loaded utilities:", uc.__file__)

## 1. Load H5 summaries

This loader avoids importing `src/functions.py`, so it can handle XGB and ENet outputs in the same notebook without module-name conflicts.

In [ ]:
from pathlib import Path
import json
import lzma
import pickle
import re
import warnings

import h5py
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


# ============================================================
# Scan all model folders
# ============================================================
results = scan_all_results(RESULT_FOLDERS)

print("\nTotal files scanned:", len(results))

if results.empty:
    raise SystemExit("No H5 files found. Check RESULT_FOLDERS at the top of the notebook.")

# Convert numeric columns
for c in [
    "test_mse",
    "test_pearson_corr",
    "test_cosine_similarity",
    "select_k",
    "selected_feature_count",
    "train_n",
    "test_n",
    "file_size_mb",
]:
    if c in results.columns:
        results[c] = pd.to_numeric(results[c], errors="coerce")

# Ensure protein is string
results["protein"] = results["protein"].astype(str)

# Build clean run labels


results["run_label"] = results.apply(make_run_label, axis=1)

# Diagnostics
print("\nLoaded summary:")
print("  Unique proteins:", results["protein"].nunique())
print("  Unique runs:", results["run_label"].nunique())
print("  Files with load errors:", results["load_error"].notna().sum())

print("\nNon-missing metric counts by source folder:")
display(
    results.groupby("source_folder")
    .agg(
        files=("file", "count"),
        proteins=("protein", "nunique"),
        pearson_nonmissing=("test_pearson_corr", lambda s: s.notna().sum()),
        cosine_nonmissing=("test_cosine_similarity", lambda s: s.notna().sum()),
        mse_nonmissing=("test_mse", lambda s: s.notna().sum()),
    )
    .reset_index()
)

print("\nPreview:")
display(results.head())

# Keep moved utility functions synchronized with notebook variables.
try:
    uc.set_module_globals(globals())
except NameError:
    pass

## 2. Metric summaries

In [ ]:
counts = (
    results.groupby(["source_folder", "normalization", "grid_preset", "select_k"], dropna=False)
    .agg(files=("file", "count"), proteins=("protein", "nunique"))
    .reset_index()
    .sort_values(["source_folder", "normalization", "select_k"])
)
print("Files and protein coverage by run:")
display(counts)

metric_summary = (
    results.groupby("run_label")[["test_pearson_corr", "test_cosine_similarity", "test_mse"]]
    .agg(["count", "mean", "median", "std"])
)
print("Metric summary by run:")
display(metric_summary)

results.to_csv(OUT_DIR / "all_model_results_summary.csv", index=False)
counts.to_csv(OUT_DIR / "model_counts_by_run.csv", index=False)
print("Saved summary CSVs in", OUT_DIR)

# Keep moved utility functions synchronized with notebook variables.
try:
    uc.set_module_globals(globals())
except NameError:
    pass

## 3. Metric distribution plots

In [ ]:
plt.rcParams.update({
    "figure.figsize": (11, 5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
})

RUNS_ORDER = (
    results.groupby("run_label")["test_pearson_corr"]
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)


boxplot_metric(results, "test_pearson_corr", "Test Pearson distribution by run", True)
plt.show()
boxplot_metric(results, "test_cosine_similarity", "Test cosine similarity distribution by run", True)
plt.show()
fig, ax = boxplot_metric(results, "test_mse", "Test MSE distribution by run", False)
q99 = results["test_mse"].quantile(0.99)
if np.isfinite(q99):
    ax.set_xlim(left=0, right=q99)
plt.show()

# Keep moved utility functions synchronized with notebook variables.
try:
    uc.set_module_globals(globals())
except NameError:
    pass

## 4. Threshold summaries

The threshold table is printed rather than plotted. For MSE, if `MSE_GOOD_THRESHOLD = None`, the cutoff is the 25th percentile of the per-protein best MSE.

In [ ]:
best_pearson = best_by_metric(results, "test_pearson_corr", higher=True)
best_cosine = best_by_metric(results, "test_cosine_similarity", higher=True)
best_mse = best_by_metric(results, "test_mse", higher=False)

if MSE_GOOD_THRESHOLD is None:
    mse_threshold = float(best_mse["test_mse"].quantile(0.25)) if len(best_mse) else np.nan
else:
    mse_threshold = float(MSE_GOOD_THRESHOLD)

print(f"Pearson good threshold: >= {PEARSON_GOOD_THRESHOLD}")
print(f"Cosine good threshold:  >= {COSINE_GOOD_THRESHOLD}")
print(f"MSE good threshold:     <= {mse_threshold:.4g}")

threshold_rows = []
for run_label, grp in results.groupby("run_label"):
    threshold_rows.append({
        "run_label": run_label,
        "proteins": grp["protein"].nunique(),
        "pearson_good_count": int((grp["test_pearson_corr"] >= PEARSON_GOOD_THRESHOLD).sum()),
        "cosine_good_count": int((grp["test_cosine_similarity"] >= COSINE_GOOD_THRESHOLD).sum()),
        "mse_good_count": int((grp["test_mse"] <= mse_threshold).sum()),
        "median_pearson": grp["test_pearson_corr"].median(),
        "median_cosine": grp["test_cosine_similarity"].median(),
        "median_mse": grp["test_mse"].median(),
    })
threshold_summary = pd.DataFrame(threshold_rows).sort_values("median_pearson", ascending=False)
display(threshold_summary)
threshold_summary.to_csv(OUT_DIR / "threshold_summary_by_run.csv", index=False)

print("Best possible per protein if we choose the best run by each metric:")
print("  Pearson:", len(best_pearson), "evaluated |", int((best_pearson["test_pearson_corr"] >= PEARSON_GOOD_THRESHOLD).sum()), "well predicted")
print("  Cosine: ", len(best_cosine), "evaluated |", int((best_cosine["test_cosine_similarity"] >= COSINE_GOOD_THRESHOLD).sum()), "well predicted")
print("  MSE:    ", len(best_mse), "evaluated |", int((best_mse["test_mse"] <= mse_threshold).sum()), "well predicted")

# Keep moved utility functions synchronized with notebook variables.
try:
    uc.set_module_globals(globals())
except NameError:
    pass

## 5. Best run per protein

In [ ]:
print("Best run per protein by Pearson:")
display(summarize_best(best_pearson, "test_pearson_corr", True, PEARSON_GOOD_THRESHOLD).head(20))
print("Best run per protein by cosine:")
display(summarize_best(best_cosine, "test_cosine_similarity", True, COSINE_GOOD_THRESHOLD).head(20))
print("Best run per protein by MSE:")
display(summarize_best(best_mse, "test_mse", False, mse_threshold).head(20))

best_pearson.to_csv(OUT_DIR / "best_per_protein_by_pearson.csv", index=False)
best_cosine.to_csv(OUT_DIR / "best_per_protein_by_cosine.csv", index=False)
best_mse.to_csv(OUT_DIR / "best_per_protein_by_mse.csv", index=False)

# Keep moved utility functions synchronized with notebook variables.
try:
    uc.set_module_globals(globals())
except NameError:
    pass

## 6. Protein partition plots

Each horizontal bar represents all proteins. Every segment shows how many proteins are best predicted by a model/run according to the chosen metric. Proteins whose best score is still below threshold are grouped into a red **Bad** segment.

In [ ]:
from pathlib import Path
from typing import Optional, Tuple
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ============================================================
# Output directory
# ============================================================
OUT_DIR = Path("./output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# Display and quality thresholds
# ============================================================
MIN_PROTEINS_TO_SHOW_AS_OWN_SEGMENT = 200

# Pearson threshold:
# If previous threshold was already >= 0.5, raise to 0.7.
_previous_pearson = globals().get("PEARSON_GOOD_THRESHOLD", None)
if _previous_pearson is not None and float(_previous_pearson) >= 0.5:
    PEARSON_GOOD_THRESHOLD = 0.70
else:
    PEARSON_GOOD_THRESHOLD = 0.50

COSINE_GOOD_THRESHOLD = float(globals().get("COSINE_GOOD_THRESHOLD", 0.70))

# MSE threshold: lower is better.
# Keep existing threshold if it exists, otherwise use median of best-MSE results.
if "mse_threshold" not in globals():
    mse_threshold = float(best_mse["test_mse"].dropna().median())

print("Good-prediction thresholds:")
print(f"  Pearson good if >= {PEARSON_GOOD_THRESHOLD:.2f}")
print(f"  Cosine  good if >= {COSINE_GOOD_THRESHOLD:.2f}")
print(f"  MSE     good if <= {mse_threshold:.4g}")
print(f"  Individual model segments shown only if they contain >= {MIN_PROTEINS_TO_SHOW_AS_OWN_SEGMENT} proteins")


# ============================================================
# Helpers
# ============================================================


# ============================================================
# Three partition plots
# ============================================================

partition_pearson = plot_partition_bar(
    best_pearson,
    metric="test_pearson_corr",
    threshold=PEARSON_GOOD_THRESHOLD,
    higher=True,
    title=f"Protein partition: best model by Pearson, good if Pearson ≥ {PEARSON_GOOD_THRESHOLD:.2f}",
    filename="partition_best_by_pearson.png",
    min_proteins_to_show=MIN_PROTEINS_TO_SHOW_AS_OWN_SEGMENT,
)

partition_cosine = plot_partition_bar(
    best_cosine,
    metric="test_cosine_similarity",
    threshold=COSINE_GOOD_THRESHOLD,
    higher=True,
    title=f"Protein partition: best model by cosine similarity, good if cosine ≥ {COSINE_GOOD_THRESHOLD:.2f}",
    filename="partition_best_by_cosine.png",
    min_proteins_to_show=MIN_PROTEINS_TO_SHOW_AS_OWN_SEGMENT,
)

partition_mse = plot_partition_bar(
    best_mse,
    metric="test_mse",
    threshold=mse_threshold,
    higher=False,
    title=f"Protein partition: best model by MSE, good if MSE ≤ {mse_threshold:.4g}",
    filename="partition_best_by_mse.png",
    min_proteins_to_show=MIN_PROTEINS_TO_SHOW_AS_OWN_SEGMENT,
)

# Keep moved utility functions synchronized with notebook variables.
try:
    uc.set_module_globals(globals())
except NameError:
    pass

## 7. Optional: inspect one best model artifact

In [ ]:
if len(best_pearson) == 0:
    print("No best result available.")
else:
    example_path = Path(best_pearson.iloc[0]["h5_path"])
    print("Example H5 selected:", example_path)
    try:
        artifact = load_artifact_from_h5(example_path)
        with h5py.File(example_path, "r") as h5:
            summary = read_json_dataset(h5, ["summary/json", "metrics/json"])
        print("Protein:", summary.get("protein", example_path.stem.split("_")[0]))
        print("Test MSE:", summary.get("test_mse"))
        print("Test Pearson:", summary.get("test_pearson_corr"))
        print("Test cosine:", summary.get("test_cosine_similarity"))
        selected = summary.get("selected_features", None)
        if selected is None and artifact is not None:
            selected = artifact.get("selected_features", [])
        selected = selected or []
        print("Selected features:", len(selected))
        print("First 20 selected features:", selected[:20])
        if artifact is not None:
            model = artifact.get("model")
            print("Artifact kind:", artifact.get("artifact_kind"))
            print("Model class:", type(model))
            print("Normalization:", artifact.get("normalization"))
            print("Params:")
            print(json.dumps(summary.get("best_params", artifact.get("params", {})), indent=2, default=str))
            if hasattr(model, "coef_"):
                weights = pd.DataFrame({"gene": artifact.get("selected_features", []), "coef": model.coef_})
                weights["abs_coef"] = weights["coef"].abs()
                weights_nonzero = weights[weights["coef"] != 0].sort_values("abs_coef", ascending=False).drop(columns="abs_coef")
                print("Total coefficients:", len(weights))
                print("Nonzero coefficients:", len(weights_nonzero))
                display(weights_nonzero.head(30))
    except Exception as e:
        print("Could not inspect model artifact:", repr(e))

# Keep moved utility functions synchronized with notebook variables.
try:
    uc.set_module_globals(globals())
except NameError:
    pass

## 8. Export compact best-model table

In [ ]:
wide_best = pd.DataFrame({"protein": sorted(results["protein"].unique())})
for name, best_df, metric in [
    ("pearson", best_pearson, "test_pearson_corr"),
    ("cosine", best_cosine, "test_cosine_similarity"),
    ("mse", best_mse, "test_mse"),
]:
    tmp = best_df[["protein", "run_label", metric, "h5_path"]].rename(columns={
        "run_label": f"best_{name}_run",
        metric: f"best_{name}_score",
        "h5_path": f"best_{name}_h5_path",
    })
    wide_best = wide_best.merge(tmp, on="protein", how="left")

wide_best.to_csv(OUT_DIR / "best_model_per_protein_all_metrics.csv", index=False)
print("Saved:", OUT_DIR / "best_model_per_protein_all_metrics.csv")
display(wide_best.head())

# Keep moved utility functions synchronized with notebook variables.
try:
    uc.set_module_globals(globals())
except NameError:
    pass